# 📈 Урок 52: Matplotlib — Візуальний фундамент

> **Мета уроку:** Зрозуміти Matplotlib як **низькорівневий рушій (low-level engine)** — навчитися не просто будувати графіки, а **думати через візуалізацію**: коли яким графіком користуватись і що саме він розкриває в даних.

---

## 🗺️ Зміст уроку

1. [Архітектура Matplotlib: Figure vs Axes
2. [Line Plot — часові ряди та тренди
3. [Bar Chart — порівняння категорій
4. [Histogram — розподіл даних
5. [Subplots — дашборди та порівняння
6. [Стилізація: кольори, маркери, теми

---

**Датасети:**
- 🌾 `wfp_food_prices_ukr.csv` — ціни на продукти харчування
- 💱 `exchange-rates_ukr.csv` — курс UAH/USD
- 📉 `poverty_ukr.csv` — соціально-економічні показники
- 🌍 `global-market-monitor.csv` — глобальний моніторинг ринку

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

DATA_DIR = 'data'

# --- Завантаження датасетів ---
df_food = pd.read_csv(
    os.path.join(DATA_DIR, 'wfp_food_prices_ukr.csv'),
    usecols=['date', 'admin1', 'market', 'category', 'commodity', 'unit', 'currency', 'price'],
    parse_dates=['date']
)

df_rates = pd.read_csv(
    os.path.join(DATA_DIR, 'exchange-rates_ukr.csv'),
    usecols=['StartDate', 'Year', 'Months', 'Value'],
    parse_dates=['StartDate']
)
df_rates.columns = ['date', 'year', 'month', 'uah_per_usd']

df_poverty = pd.read_csv(os.path.join(DATA_DIR, 'poverty_ukr.csv'))

# Тільки гривневі ціни
df_uah = df_food.query("currency == 'UAH'").copy()

print(f"✅ Продовольчі ціни:  {df_food.shape[0]:,} рядків")
print(f"✅ Курс UAH/USD:       {df_rates.shape[0]} рядків")
print(f"✅ Соц. показники:     {df_poverty.shape[0]} рядків")

---

## 1. Архітектура Matplotlib: Figure vs Axes 🏗️ <a id='1-architecture'></a>

### Matplotlib = low-level engine

Matplotlib — це **фундаментальний рушій** візуалізації в Python. Він дає повний контроль над кожним елементом: від товщини лінії до позиції легенди. Саме на його базі побудовані `pandas.plot()` та `seaborn`.

### Сувора ієрархія об'єктів

У Matplotlib **немає «просто графіка»** — є ієрархія:

```
Figure  ← «аркуш паперу» / головний контейнер
└── Axes  ← область побудови (ВСЯ система координат)
    ├── XAxis / YAxis  ← осі з мітками
    ├── Title, Legend
    └── Line2D, Patch, Text...  ← графічні примітиви
```

> ⚠️ **Увага:** `Axes` у Matplotlib — це НЕ «осі X та Y». Це цілий об'єкт-графік зі своєю системою координат. Одна `Figure` може містити багато `Axes`.

### Два стилі роботи: pyplot vs OOP

| Стиль | Синтаксис | Коли використовувати |
|---|---|---|
| **pyplot** (швидко) | `plt.plot(x, y)` | Простий одиночний графік |
| **OOP** (правильно) | `fig, ax = plt.subplots()` → `ax.plot(x, y)` | Subplots, складна кастомізація |

In [ ]:
# --- Демонстрація: anatomy of a Figure ---
fig, ax = plt.subplots(figsize=(10, 6))

print(f"Тип 'fig': {type(fig)}")
print(f"Тип 'ax' : {type(ax)}")

x = [1, 2, 3, 4, 5]
y = [10, 25, 18, 35, 28]

# Малюємо на конкретному Axes об'єкті
line = ax.plot(x, y, color='steelblue', linewidth=2, marker='o', label='Значення')

# Кожен рівень ієрархії має свій метод
ax.set_title('Анатомія Figure та Axes', fontsize=14, fontweight='bold')
ax.set_xlabel('Вісь X (незалежна змінна)')
ax.set_ylabel('Вісь Y (залежна змінна)')
ax.legend()
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

print(f"\nLine2D об'єкт: {line[0]}")
print(f"Figure розмір: {fig.get_size_inches()} дюймів")

In [ ]:
# --- rcParams: глобальний стан рушія ---
# Всі налаштування за замовчуванням зберігаються в словнику rcParams
print("=== Деякі налаштування rcParams ===")
interesting_keys = [
    'figure.figsize', 'lines.linewidth', 'font.size',
    'axes.facecolor', 'axes.grid'
]
for key in interesting_keys:
    print(f"  {key}: {mpl.rcParams[key]}")

print(f"\n💡 plt.style.use() перезаписує одразу сотні цих значень")
print(f"Доступні стилі: {plt.style.available[:8]} ...")

---

## 2. Line Plot — часові ряди та тренди 📉 <a id='2-lineplot'></a>

### Навіщо лінійний графік?

Line plot — базовий інструмент для **часових рядів (time series)**. Коли вісь X є безперервною (дні, місяці, роки), він дозволяє миттєво зчитувати:

| Що шукаємо | Що бачимо на графіку |
|---|---|
| **Тренд** | Загальний напрямок кривої (вгору/вниз) |
| **Циклічність** | Повторювані патерни (сезонність) |
| **Аномалія** | Різкий стрибок або провал, що вибивається з ряду |

### Що відбувається під капотом `plt.plot()`

Коли ви викликаєте `ax.plot(x, y)`, Matplotlib:
1. Бере масиви `x` і `y`, поєднуючи їх у пари координат
2. Створює графічний примітив **`Line2D`** — набір прямих відрізків між точками
3. Якщо передати лише `plot(y)` — вісь X буде автоматично `0, 1, 2, ...`

Плавна крива — це **ілюзія**: чим більше точок, тим плавніше виглядає лінія.

### Параметри кастомізації

```python
ax.plot(x, y,
    color='steelblue',    # колір
    linestyle='--',       # '-', '--', ':', '-.'  
    linewidth=2,          # товщина
    marker='o',           # маркер на точках
    alpha=0.8             # прозорість
)

# Або короткий формат-рядок:
ax.plot(x, y, 'ro--')   # червоний, круглі маркери, штрихова
```

In [ ]:
# --- Підготовка даних: курс гривні до долара ---
df_rates['рік'] = df_rates['date'].dt.year
yearly_rate = (
    df_rates
    .groupby('рік')['uah_per_usd']
    .mean()
    .reset_index()
)
yearly_rate = yearly_rate[yearly_rate['рік'] >= 1997]
# --- Line plot: курс UAH/USD по роках ---
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(
    yearly_rate['рік'],
    yearly_rate['uah_per_usd'],
    color='#1a6faf',
    linewidth=2.5,
    marker='o',
    markersize=5,
    markerfacecolor='white',   # заливка маркера
    markeredgewidth=2,         # контур маркера
    label='Середній річний курс'
)

# Додаємо вертикальну позначку для 2022
ax.axvline(x=2022, color='crimson', linestyle='--', linewidth=1.5, alpha=0.7, label='24.02.2022')
ax.axvspan(2022, yearly_rate['рік'].max(), alpha=0.05, color='red')

ax.set_title('Курс гривні до долара США (1997–2024)', fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Рік')
ax.set_ylabel('UAH за 1 USD')
ax.legend()
ax.grid(True, linestyle=':', alpha=0.5)
ax.set_xlim(yearly_rate['рік'].min(), yearly_rate['рік'].max())

fig.tight_layout()
plt.show()

In [ ]:
# --- Кілька ліній: порівняння цін на різні продукти ---
# Підготовка: середня ціна по роках для топ-3 товарів
top_commodities = ['Bread (wheat)', 'Potatoes', 'Oil (sunflower)']

df_top = (
    df_uah
    .query("commodity in @top_commodities and admin1.isna()")
    .assign(рік=lambda x: x['date'].dt.year)
    .groupby(['рік', 'commodity'])['price']
    .mean()
    .round(2)
    .reset_index()
)

colors = {'Bread (wheat)': '#e07b39', 'Potatoes': '#5ba85f', 'Oil (sunflower)': '#3b82c4'}
markers = {'Bread (wheat)': 's', 'Potatoes': '^', 'Oil (sunflower)': 'o'}

fig, ax = plt.subplots(figsize=(12, 5))

# plt.plot() можна викликати кілька разів — лінії накладаються
for commodity, group in df_top.groupby('commodity'):
    ax.plot(
        group['рік'], group['price'],
        color=colors[commodity],
        marker=markers[commodity],
        linewidth=2,
        markersize=5,
        label=commodity
    )

ax.axvline(x=2022, color='gray', linestyle='--', linewidth=1, alpha=0.6)
ax.set_title('Динаміка цін на основні продукти в Україні (UAH)', fontsize=14, fontweight='bold')
ax.set_xlabel('Рік')
ax.set_ylabel('Середня ціна (UAH / кг або одиниця)')
ax.legend(title='Продукт')
ax.grid(True, linestyle=':', alpha=0.4)

fig.tight_layout()
plt.show()

### 🎯 Міні-завдання 1

1. Побудуй line plot середньої ціни на `Milk` по місяцях (не по роках) — чи є сезонний патерн?
2. Додай горизонтальну лінію (`ax.axhline`) на рівні середньої ціни за весь період
3. Виділь зону після 2022 року кольоровим прямокутником (`ax.axvspan`)

> **Очікуваний інсайт:** Лінійний графік миттєво показує, коли сталися структурні зрушення в ціні — ці точки збігаються з ключовими подіями (девальвація 2014–2015, COVID-19, 2022).

In [ ]:
# 🎯 Ваш код тут:


---

## 3. Bar Chart — порівняння категорій 📊 <a id='3-barchart'></a>

### Навіщо стовпчаста діаграма?

Bar chart призначений для **порівняння дискретних категорій**, а не для відображення часу. Вісь X не має числової неперервності — це набір незалежних категорій.

### Що відбувається під капотом `plt.bar()`

`plt.bar(x, height)` фізично малює **набір прямокутників (`patches`)** на координатній площині. Ти маєш повний контроль:

| Параметр | Призначення |
|---|---|
| `width` | Ширина стовпця (за замовчуванням 0.8) |
| `color` / `edgecolor` | Заливка та контур |
| `bottom` | «Дно» стовпця — для **stacked bar** |
| `yerr` | «Вуса» похибок поверх стовпців |

### Вертикальні vs Горизонтальні (`plt.barh`)

- **`plt.bar()`** — вертикальні. Добре для <10 категорій з короткими назвами
- **`plt.barh()`** — горизонтальні. Рятує, коли категорій багато або назви довгі

### Grouped та Stacked Bar Charts

Matplotlib не будує їх «магічно» — потребує **ручного розрахунку координат**:
- **Grouped:** зменшуємо `width` і зсуваємо кожну серію на `±width/2`
- **Stacked:** передаємо `bottom=попередня_серія` для кожного наступного шару

> 💡 **Зв'язок з Pandas:** `df.plot.bar()` робить всі ці розрахунки за тебе, передаючи готові координати в Matplotlib.

In [ ]:
# --- Підготовка: середня ціна за категорією у 2023 році ---
avg_by_cat_2023 = (
    df_uah
    .query("date.dt.year == 2023")
    .groupby('category')['price']
    .mean()
    .round(1)
    .sort_values(ascending=True)  # для barh краще сортувати
)

# --- Horizontal Bar Chart ---
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.barh(
    avg_by_cat_2023.index,
    avg_by_cat_2023.values,
    color='#3b82c4',
    edgecolor='white',
    height=0.6
)

# Додаємо значення на кінці стовпців
for bar in bars:
    width = bar.get_width()
    ax.text(
        width + 1, bar.get_y() + bar.get_height() / 2,
        f'{width:.1f}',
        va='center', ha='left', fontsize=9, color='#333'
    )

ax.set_title('Середня ціна продуктів за категорією (2023, UAH)', fontsize=13, fontweight='bold')
ax.set_xlabel('Середня ціна (UAH)')
ax.set_xlim(0, avg_by_cat_2023.max() * 1.15)
ax.grid(True, axis='x', linestyle=':', alpha=0.5)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# --- Grouped Bar Chart: порівняння цін до/після 2022 ---
categories = ['cereals and tubers', 'vegetables and fruits', 'oil and fats', 'meat, fish and eggs']

avg_before = (
    df_uah
    .query("date.dt.year.between(2019, 2020) and category in @categories")
    .groupby('category')['price'].mean().round(1)
    .reindex(categories)
)
avg_after = (
    df_uah
    .query("date.dt.year.between(2022, 2024) and category in @categories")
    .groupby('category')['price'].mean().round(1)
    .reindex(categories)
)

x = np.arange(len(categories))
width = 0.35  # ширина одного стовпця

fig, ax = plt.subplots(figsize=(11, 5))

# Зсуваємо кожну серію: -width/2 та +width/2
bars1 = ax.bar(x - width/2, avg_before.values, width, label='2019–2021 (до)', color='#5ba85f', edgecolor='white')
bars2 = ax.bar(x + width/2, avg_after.values,  width, label='2022–2024 (після)', color='#e07b39', edgecolor='white')

# Підписи зростання
for b1, b2 in zip(bars1, bars2):
    pct = (b2.get_height() / b1.get_height() - 1) * 100
    ax.text(
        b2.get_x() + b2.get_width() / 2,
        b2.get_height() + 1,
        f'+{pct:.0f}%', ha='center', va='bottom', fontsize=9, color='crimson', fontweight='bold'
    )

ax.set_title('Ціни на продукти: до та після 2022 року (UAH)', fontsize=13, fontweight='bold')
ax.set_xticks(x)
short_labels = ['Зернові/крупи', 'Овочі/фрукти', 'Олія/жири', "М'ясо/риба/яйця"]
ax.set_xticklabels(short_labels, fontsize=10)
ax.set_ylabel('Середня ціна (UAH)')
ax.legend()
ax.grid(True, axis='y', linestyle=':', alpha=0.4)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
# --- Stacked Bar Chart: структура цін по роках ---
# Частка кожної категорії в загальному обсязі спостережень
years = [2019, 2020, 2021, 2022, 2023]
cats_short = ['cereals and tubers', 'vegetables and fruits', 'oil and fats']

pivot = (
    df_uah
    .assign(рік=lambda x: x['date'].dt.year)
    .query("рік in @years and category in @cats_short")
    .groupby(['рік', 'category'])['price']
    .mean()
    .unstack(fill_value=0)
    .round(1)
)

colors_stack = ['#3b82c4', '#5ba85f', '#e07b39']
bottom = np.zeros(len(pivot))

fig, ax = plt.subplots(figsize=(9, 5))

for col, color in zip(pivot.columns, colors_stack):
    ax.bar(pivot.index, pivot[col], bottom=bottom, label=col, color=color, width=0.6, edgecolor='white')
    bottom += pivot[col].values  # наступний шар починається там, де закінчився попередній

ax.set_title('Накопичені середні ціни по категоріях (UAH)', fontsize=13, fontweight='bold')
ax.set_xlabel('Рік')
ax.set_ylabel('Сума середніх цін (UAH)')
ax.legend(loc='upper left', fontsize=9)
ax.grid(True, axis='y', linestyle=':', alpha=0.4)

fig.tight_layout()
plt.show()
print("💡 Параметр 'bottom' — ключ до stacked bar. Кожен шар 'стоїть' на попередньому.")

### 🎯 Міні-завдання 2

1. Побудуй горизонтальний bar chart топ-10 найдорожчих товарів у 2024 році
2. Додай планки похибок (`yerr=стандартне_відхилення`) до одного з графіків
3. Порівняй ціни в Київській та Львівській областях у 2023 році

> **Очікуваний інсайт:** Grouped bar миттєво показує нерівномірне зростання — деякі категорії подорожчали на 40–60%, інші — менш суттєво.

In [ ]:
# 🎯 Ваш код тут:


---

## 4. Histogram — розподіл даних 📐 <a id='4-histogram'></a>

### Навіщо гістограма?

Гістограма — це інструмент для розуміння **форми розподілу** числових даних. Аналітик зчитує три речі:

| Що шукаємо | Що означає |
|---|---|
| **Центральна тенденція** | Де зосереджена основна маса значень |
| **Асиметрія (Skewness)** | Хвіст вправо = позитивна асиметрія (більшість дешево, є дорогі) |
| **Викиди (Outliers)** | Ізольовані стовпці далеко від основної маси |

### Архітектурна особливість: `plt.hist()` = обчислення + малювання

На відміну від `plot()` або `bar()`, функція `hist()` **сама виконує математику**:
1. Розбиває діапазон даних на `bins` рівних інтервалів
2. Рахує кількість значень у кожному інтервалі
3. Повертає кортеж `(n, bins, patches)` — частоти, межі кошиків і самі прямокутники

### Проблема bins = «роздільна здатність»

- **Мало bins** → втрата деталей, грубі блоки
- **Забагато bins** → шум, «шипастий» графік
- **Рішення:** KDE (Kernel Density Estimate) — неперервна крива щільності без проблеми бінів

### Відмінність від Bar Chart

```
Bar Chart:   вісь X = категорії (рядки)  → стовпці з ЗАЗОРАМИ
Histogram:   вісь X = числова шкала     → стовпці ВПРИТУЛ (зазор = відсутність даних)
```

In [ ]:
# --- Базова гістограма: розподіл цін на продукти ---
prices_2023 = df_uah.query("date.dt.year == 2023 and price < 200")['price']

fig, ax = plt.subplots(figsize=(10, 5))

# hist() повертає (n=частоти, bins=межі, patches=прямокутники)
n, bins, patches = ax.hist(
    prices_2023,
    bins=30,
    color='#3b82c4',
    edgecolor='white',
    linewidth=0.5,
    alpha=0.85
)

# Вертикальні лінії для середнього та медіани
mean_price = prices_2023.mean()
median_price = prices_2023.median()
ax.axvline(mean_price,  color='crimson', linestyle='--', linewidth=2, label=f'Середнє: {mean_price:.1f} UAH')
ax.axvline(median_price, color='orange',  linestyle='-.',  linewidth=2, label=f'Медіана: {median_price:.1f} UAH')

ax.set_title('Розподіл цін на продукти в Україні (2023, UAH < 200)', fontsize=13, fontweight='bold')
ax.set_xlabel('Ціна (UAH)')
ax.set_ylabel('Кількість спостережень')
ax.legend()
ax.grid(True, axis='y', linestyle=':', alpha=0.4)

fig.tight_layout()
plt.show()

print(f"Асиметрія: {prices_2023.skew():.2f} (позитивна → хвіст вправо = є дорогі викиди)")

In [ ]:
# --- Демонстрація: вплив кількості bins на сприйняття ---
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

bins_options = [5, 30, 150]
titles = ['Занадто мало (bins=5)', 'Оптимально (bins=30)', 'Занадто багато (bins=150)']

for ax, bins_n, title in zip(axes, bins_options, titles):
    ax.hist(prices_2023, bins=bins_n, color='#3b82c4', edgecolor='white', alpha=0.85)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Ціна (UAH)')
    ax.grid(True, axis='y', linestyle=':', alpha=0.4)

axes[0].set_ylabel('Кількість')
fig.suptitle('Вплив кількості bins на гістограму', fontsize=13, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# --- KDE як альтернатива гістограмі ---
# KDE не залежить від вибору bins — показує неперервну криву щільності

bread_prices = df_uah.query("commodity.str.contains('Bread') and price < 100")['price']
potato_prices = df_uah.query("commodity == 'Potatoes' and price < 100")['price']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Ліворуч: гістограма
axes[0].hist(bread_prices, bins=30, alpha=0.6, color='#e07b39', label='Хліб', density=True)
axes[0].hist(potato_prices, bins=30, alpha=0.6, color='#5ba85f', label='Картопля', density=True)
axes[0].set_title('Гістограма (bins=30)', fontweight='bold')
axes[0].set_xlabel('Ціна (UAH)')
axes[0].set_ylabel('Щільність')
axes[0].legend()

# Праворуч: KDE через pandas
bread_prices.plot.kde(ax=axes[1], color='#e07b39', linewidth=2.5, label='Хліб')
potato_prices.plot.kde(ax=axes[1], color='#5ba85f', linewidth=2.5, label='Картопля')
axes[1].set_title('KDE — крива щільності (без проблеми bins)', fontweight='bold')
axes[1].set_xlabel('Ціна (UAH)')
axes[1].set_xlim(0, 100)
axes[1].legend()
axes[1].grid(True, linestyle=':', alpha=0.4)

fig.suptitle('Гістограма vs KDE: два способи побачити розподіл', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

print("💡 KDE не потребує вибору bins — показує ту ж форму розподілу, але без артефактів")

### 🎯 Міні-завдання 3

1. Побудуй гістограму розподілу курсу UAH/USD за весь час
2. Зафарбуй стовпці **після 2014 року** в інший колір, ніж до 2014 (hint: використовуй два виклики `hist()` на одних осях)
3. Порівняй асиметрію (`skew()`) розподілу цін на хліб та на м'ясо

> **Очікуваний інсайт:** Розподіл курсу UAH/USD має виражену позитивну асиметрію і навіть **бімодальний** характер — до і після великих девальвацій курс концентрувався навколо зовсім різних значень.

In [ ]:
# 🎯 Ваш код тут:


---

## 5. Subplots — дашборди та порівняння 🖥️ <a id='5-subplots'></a>

### Навіщо subplots?

Один графік відповідає на одне питання. Дашборд — на декілька одночасно. Subplots дозволяють розмістити кілька незалежних `Axes` на одному `Figure`.

### Анатомія: `plt.subplots(nrows, ncols)`

```python
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
#           ↑ Figure    ↑ масив Axes 2×2

axes[0, 0].plot(...)  # верхній лівий
axes[0, 1].bar(...)   # верхній правий
axes[1, 0].hist(...)  # нижній лівий
axes[1, 1].plot(...)  # нижній правий
```

### Аналітичні суперсили subplots

| Параметр | Ефект |
|---|---|
| `sharex=True` | Спільна вісь X — чесне порівняння часових рядів |
| `sharey=True` | Спільна вісь Y — чесне порівняння масштабів |
| `fig.tight_layout()` | Автоматичні відступи, щоб підписи не накладались |

### Розширений контроль: GridSpec

Якщо рівномірна сітка не підходить — `GridSpec` дозволяє графікам займати кілька комірок (аналог `colspan`/`rowspan` у HTML-таблицях):

```python
from matplotlib.gridspec import GridSpec
gs = GridSpec(2, 3)  # сітка 2×3
ax1 = fig.add_subplot(gs[0, :])   # перший рядок — всі 3 колонки
ax2 = fig.add_subplot(gs[1, 0:2]) # другий рядок — перші 2 колонки
ax3 = fig.add_subplot(gs[1, 2])   # другий рядок — остання колонка
```

In [ ]:
# --- Підготовка даних для дашборду ---
df_uah['рік'] = df_uah['date'].dt.year
df_rates['рік'] = df_rates['date'].dt.year

bread_trend = (
    df_uah.query("commodity.str.contains('Bread') and admin1.isna()")
    .groupby('рік')['price'].mean().round(2)
)
potato_trend = (
    df_uah.query("commodity == 'Potatoes' and admin1.isna()")
    .groupby('рік')['price'].mean().round(2)
)
rate_trend = (
    df_rates[df_rates['рік'] > 1996]
    .groupby('рік')['uah_per_usd']
    .mean()
    .round(2)
)
cat_avg_2023 = (
    df_uah.query("рік == 2023")
    .groupby('category')['price'].mean().round(1)
    .sort_values(ascending=False)
)

In [ ]:
# --- Дашборд 2×2: sharex для синхронізованих часових рядів ---
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Аналіз продовольчої ситуації в Україні', fontsize=15, fontweight='bold', y=1.01)

# --- [0,0] Курс UAH/USD ---
axes[0, 0].plot(rate_trend.index, rate_trend.values, color='#1a6faf', linewidth=2.5, marker='o', markersize=4)
axes[0, 0].axvline(2022, color='crimson', linestyle='--', alpha=0.6)
axes[0, 0].set_title('Курс UAH/USD', fontweight='bold')
axes[0, 0].set_ylabel('UAH за USD')
axes[0, 0].grid(True, linestyle=':', alpha=0.4)

# --- [0,1] Ціна хліба ---
axes[0, 1].plot(bread_trend.index, bread_trend.values, color='#e07b39', linewidth=2.5, marker='s', markersize=4)
axes[0, 1].axvline(2022, color='crimson', linestyle='--', alpha=0.6)
axes[0, 1].set_title('Ціна хліба (UAH)', fontweight='bold')
axes[0, 1].set_ylabel('UAH')
axes[0, 1].grid(True, linestyle=':', alpha=0.4)

# --- [1,0] Гістограма цін на картоплю ---
potato_data = df_uah.query("commodity == 'Potatoes' and price < 80")['price']
axes[1, 0].hist(potato_data, bins=35, color='#5ba85f', edgecolor='white', alpha=0.85)
axes[1, 0].axvline(potato_data.mean(), color='crimson', linestyle='--', linewidth=2,
                    label=f'Середнє: {potato_data.mean():.1f}')
axes[1, 0].set_title('Розподіл цін на картоплю', fontweight='bold')
axes[1, 0].set_xlabel('Ціна (UAH/кг)')
axes[1, 0].set_ylabel('Кількість')
axes[1, 0].legend(fontsize=9)
axes[1, 0].grid(True, axis='y', linestyle=':', alpha=0.4)

# --- [1,1] Bar: середня ціна за категорією 2023 ---
axes[1, 1].barh(cat_avg_2023.index, cat_avg_2023.values, color='#3b82c4', height=0.6, edgecolor='white')
axes[1, 1].set_title('Середня ціна за категорією (2023)', fontweight='bold')
axes[1, 1].set_xlabel('Ціна (UAH)')
axes[1, 1].grid(True, axis='x', linestyle=':', alpha=0.4)

fig.tight_layout()  # рятує від накладання підписів
plt.show()

In [ ]:
# --- sharex=True: синхронізована вісь X для порівняння ---
# Порівнюємо курс і ціну хліба — чи рухаються синхронно?

common_years = sorted(set(bread_trend.index) & set(rate_trend.index))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

ax1.plot(common_years, [bread_trend[y] for y in common_years], color='#e07b39', linewidth=2.5, marker='o', markersize=4)
ax1.set_ylabel('Ціна хліба (UAH)', fontsize=11)
ax1.set_title('Синхронізація: курс гривні та ціни на хліб', fontsize=13, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.4)
ax1.axvline(2022, color='gray', linestyle='--', alpha=0.5)

ax2.plot(common_years, [rate_trend[y] for y in common_years], color='#1a6faf', linewidth=2.5, marker='s', markersize=4)
ax2.set_ylabel('Курс UAH/USD', fontsize=11)
ax2.set_xlabel('Рік', fontsize=11)
ax2.grid(True, linestyle=':', alpha=0.4)
ax2.axvline(2022, color='gray', linestyle='--', alpha=0.5)

# sharex=True: підписи X тільки на нижньому графіку
fig.tight_layout()
plt.show()

corr = pd.Series([bread_trend.get(y, None) for y in common_years]).corr(
       pd.Series([rate_trend.get(y, None) for y in common_years]))
print(f"\n📊 Кореляція між ціною хліба та курсом UAH/USD: {corr:.3f}")
print("💡 sharex=True: мітки на осі X відображаються лише на нижньому графіку — ніякого дублювання")

In [ ]:
# --- GridSpec: нерівномірна сітка ---
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(14, 7))
fig.suptitle('GridSpec: дашборд з нерівномірною сіткою', fontsize=13, fontweight='bold')

gs = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

# Великий графік — займає всі 3 колонки першого рядка
ax_main = fig.add_subplot(gs[0, :])  # рядок 0, всі колонки
ax_main.plot(rate_trend.index, rate_trend.values, color='#1a6faf', linewidth=2)
ax_main.set_title('Курс UAH/USD (головний графік)', fontweight='bold')
ax_main.grid(True, linestyle=':', alpha=0.4)

# Три маленьких графіки — кожен займає 1 колонку другого рядка
small_items = [
    ('Bread (wheat)', '#e07b39'),
    ('Potatoes', '#5ba85f'),
    ('Oil (sunflower)', '#9b59b6')
]
for col, (commodity, color) in enumerate(small_items):
    ax = fig.add_subplot(gs[1, col])
    trend = df_uah.query("commodity == @commodity and admin1.isna()").groupby('рік')['price'].mean()
    ax.plot(trend.index, trend.values, color=color, linewidth=2)
    ax.set_title(commodity.split(' ')[0], fontsize=10, fontweight='bold')
    ax.grid(True, linestyle=':', alpha=0.4)
    ax.tick_params(axis='x', labelsize=8)

plt.show()
print("💡 GridSpec дозволяє будь-якому Axes займати кілька рядків або стовпців сітки")

### 🎯 Міні-завдання 4

1. Створи дашборд 3×1 з `sharex=True`, де зверху — курс, в середині — ціна олії, знизу — ціна м'яса
2. Зроби один з підграфіків більшим, використовуючи `GridSpec` з `colspan=2`
3. Додай анотацію (`ax.annotate()`) до піку на одному з графіків

> **Очікуваний інсайт:** `sharex=True` буквально «синхронізує» погляд — ти одразу бачиш, що стрибок курсу у 2022 спричинив синхронне зростання цін на імпортовані товари.

In [ ]:
# 🎯 Ваш код тут:


---

## 6. Стилізація: кольори, маркери, теми 🎨 <a id='6-styling'></a>

### Три рівні стилізації

| Рівень | Інструмент | Охоплення |
|---|---|---|
| **Елемент** | Параметри `plot()`: `color`, `lw`, `marker` | Один графік |
| **Ефективний** | Формат-рядки `'ro--'` | Один графік (швидко) |
| **Глобальний** | `plt.style.use()` → `rcParams` | Весь сеанс |

### Система кольорів: від простого до точного

```python
# 1. Базові літери (спадок від MATLAB):
#    'b'=синій, 'g'=зелений, 'r'=червоний, 'k'=чорний, 'w'=білий

# 2. Іменовані CSS кольори:
color='steelblue'  # 'darkred', 'seagreen', 'orange'...

# 3. HEX-код:
color='#1a6faf'

# 4. RGB-кортеж (0–1, НЕ 0–255!):
color=(26/255, 111/255, 175/255)  # те ж #1a6faf

# 5. RGBA (з прозорістю):
color=(26/255, 111/255, 175/255, 0.7)
```

### Формат-рядки: синтаксичний цукор

```python
# Структура: '[color][marker][linestyle]'
'ro--'  # r=червоний, o=коло, --=штрихова
'ks:'   # k=чорний, s=квадрат, :=пунктирна
'b^'    # b=синій, ^=трикутник, без лінії (scatter!)
```

### Глобальні теми: `rcParams` під капотом

Команда `plt.style.use('ggplot')` миттєво **перезаписує сотні налаштувань** у глобальному словнику `matplotlib.rcParams`. Це змінює фон, шрифти, колірні палітри, товщини ліній — все за один виклик.

In [ ]:
# --- Демонстрація: система кольорів та маркерів ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x = np.linspace(0, 10, 20)

# Ліворуч: різні способи задати колір
color_demos = [
    ('r',                              'Базова літера: r'),
    ('steelblue',                      'Назва: steelblue'),
    ('#e07b39',                        'HEX: #e07b39'),
    ((26/255, 111/255, 175/255),       'RGB tuple (0–1): (26,111,175)'),
    ((0.1, 0.7, 0.2, 0.5),            'RGBA (з alpha=0.5)'),
]
for i, (color, label) in enumerate(color_demos):
    axes[0].plot(x, x + i * 2, color=color, linewidth=3, label=label)

axes[0].set_title('Способи задати колір', fontweight='bold')
axes[0].legend(fontsize=8)
axes[0].grid(True, linestyle=':', alpha=0.3)

# Праворуч: маркери та стилі ліній через формат-рядки
format_strings = ['ro-', 'bs--', 'g^:', 'k*-.', 'mD']
for i, fmt in enumerate(format_strings):
    axes[1].plot(x[::4], x[::4] + i * 2, fmt, markersize=8, linewidth=2, label=f"'{fmt}'")

axes[1].set_title('Формат-рядки: [color][marker][linestyle]', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].grid(True, linestyle=':', alpha=0.3)

fig.tight_layout()
plt.show()

In [ ]:
# --- Демонстрація глобальних тем ---
# Той самий графік у 4 різних стилях

styles = ['default', 'ggplot', 'seaborn-v0_8', 'fivethirtyeight']
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes_flat = axes.flatten()

years = list(bread_trend.index)
bread_vals = [bread_trend[y] for y in years]
potato_vals = [potato_trend.get(y, None) for y in years]

for ax, style in zip(axes_flat, styles):
    with plt.style.context(style):  # застосовуємо стиль тільки всередині блоку
        ax.plot(years, bread_vals, label='Хліб', linewidth=2, marker='o', markersize=4)
        ax.plot(years, potato_vals, label='Картопля', linewidth=2, marker='s', markersize=4)
        ax.set_title(f"style='{style}'", fontweight='bold')
        ax.legend(fontsize=8)
        ax.set_xlabel('Рік', fontsize=9)
        ax.set_ylabel('UAH', fontsize=9)

fig.suptitle('Один графік — чотири теми (rcParams)', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()
print(f"\nДоступно стилів: {len(plt.style.available)}")
print(f"Всі стилі: {plt.style.available}")

In [ ]:
# --- Фінальний графік: повна кастомізація ---
# Застосовуємо всі прийоми разом: стиль, кольори, анотації, спайни

with plt.style.context('seaborn-v0_8-whitegrid'):
    fig, ax = plt.subplots(figsize=(13, 6))

    # Основна лінія курсу
    ax.plot(rate_trend.index, rate_trend.values,
            color='#1a6faf', linewidth=3, marker='o',
            markersize=6, markerfacecolor='white',
            markeredgewidth=2, zorder=3)

    # Заливка під кривою
    ax.fill_between(rate_trend.index, rate_trend.values, alpha=0.1, color='#1a6faf')

    # Виділення ключових подій
    events = {2014: ('Майдан/початок\nдевальвації', 'orange'), 2022: ('Повномасштабне\nvторгнення', 'crimson')}
    for year, (label, color) in events.items():
        if year in rate_trend.index:
            ax.axvline(year, color=color, linestyle='--', linewidth=1.5, alpha=0.7)
            ax.annotate(
                label,
                xy=(year, rate_trend[year]),
                xytext=(year + 0.5, rate_trend[year] + 4),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5),
                fontsize=9, color=color, fontweight='bold'
            )

    # Видалення зайвих спайнів (рамок)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    ax.set_title('Курс UAH/USD: 1996–2024\nІсторія девальвації через дані',
                 fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Рік', fontsize=12)
    ax.set_ylabel('UAH за 1 USD', fontsize=12)
    ax.set_xlim(rate_trend.index.min() - 0.5, rate_trend.index.max() + 0.5)

    fig.tight_layout()
    plt.show()

### 🎯 Фінальне завдання: твій дашборд

Створи власний **аналітичний дашборд** у стилі `ggplot` або `fivethirtyeight` з 4 підграфіками:

1. **Line plot** — динаміка цін на 2 товари (на вибір) по роках
2. **Bar chart** — топ-5 найдорожчих товарів у поточному році
3. **Histogram** — розподіл цін однієї категорії + лінія середнього
4. **Line plot** — курс UAH/USD з виділеними ключовими роками

Вимоги:
- Загальний заголовок через `fig.suptitle()`
- Кастомні кольори (не за замовчуванням)
- `fig.tight_layout()` щоб не накладались підписи
- Анотація (`ax.annotate`) хоча б на одному графіку

> **Очікуваний інсайт:** Візуалізація — це не прикраса. Це **мова для передачі інсайту**. Один правильно побудований дашборд передає те, що потребувало б 3 сторінок тексту.

In [ ]:
# 🎯 Ваш дашборд тут:


---

## 🏁 Підсумок уроку

### Коли який графік?

| Тип графіку | Питання, на яке відповідає |
|---|---|
| **Line plot** | Як змінювалась величина з часом? Є тренд? |
| **Bar chart** | Яка категорія найбільша/найменша? |
| **Histogram** | Як розподілені дані? Є асиметрія/викиди? |
| **Subplots** | Як кілька величин пов'язані між собою? |

### Ключові архітектурні концепції

| Концепція | Суть |
|---|---|
| **Figure / Axes** | Figure = полотно, Axes = графік. OOP-стиль — стандарт |
| **Line2D / Patch** | Всі графіки — це набір графічних примітивів |
| **rcParams** | Глобальний стан рушія. `plt.style.use()` = масова заміна налаштувань |
| **sharex/sharey** | Чесне порівняння — один масштаб для всіх підграфіків |
| **tight_layout** | Автоматичні відступи між Axes |
| **GridSpec** | Нерівномірна сітка — для складних дашбордів |
| **KDE** | Альтернатива гістограмі без проблеми bins |

### Наступний крок

У **Уроці 52 (частина 3)** ми переходимо до **Plotly Dash** — інтерактивні дашборди у браузері!

---
*📚 Урок 52 — Модуль 3: Data Analysis & Visualization*